# EDA: Flood Probability Prediction

## Методология победителя

Победитель обнаружил:
1. **Данные = суммы Poisson распределений**
2. **sum** — самый важный признак
3. **Magic feature:** `groupby('sum')['target'].std()`
4. **Target** можно дискретизировать: `(target * 400).astype(int16)`
5. **Sorted features** и **count features** полезны

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 6)

print('✅ Библиотеки загружены')

In [ ]:
# Загрузка
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
print(f'\nColumns: {train.columns.tolist()}')
print(f'\nTarget stats:')
print(train['FloodProbability'].describe())

## 1. Распределение таргета (дискретизация от победителя)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Continuous
axes[0].hist(train['FloodProbability'], bins=100, edgecolor='black', alpha=0.7)
axes[0].set_title('FloodProbability (непрерывный)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('FloodProbability')
axes[0].set_ylabel('Частота')
axes[0].grid(alpha=0.3)

# Discrete (как у победителя!)
target_discrete = (train['FloodProbability'] * 400).astype(np.int16)
axes[1].hist(target_discrete, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('Target * 400 (дискретный для стратификации!)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Target * 400')
axes[1].set_ylabel('Частота')
axes[1].grid(alpha=0.3)

# Q-Q plot
stats.probplot(train['FloodProbability'], dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot (Normal)', fontsize=14, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Дискретный таргет:')
print(f'   Уникальных: {target_discrete.nunique()}')
print(f'   Диапазон: [{target_discrete.min()}, {target_discrete.max()}]')

## 2. Анализ SUM (самый важный признак!)

In [ ]:
original_features = [c for c in train.columns if c not in ['id', 'FloodProbability']]

# Создаем sum
train['sum'] = train[original_features].sum(axis=1)
test['sum'] = test[original_features].sum(axis=1)

# Корреляция с таргетом
corr = train[['sum', 'FloodProbability']].corr().iloc[0, 1]

print(f'🔥 КОРРЕЛЯЦИЯ sum vs FloodProbability: {corr:.6f}')

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter
axes[0].scatter(train['sum'], train['FloodProbability'], alpha=0.1, s=1)
axes[0].set_xlabel('sum', fontsize=12, fontweight='bold')
axes[0].set_ylabel('FloodProbability', fontsize=12, fontweight='bold')
axes[0].set_title(f'sum vs FloodProbability (R={corr:.4f})', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Hexbin
axes[1].hexbin(train['sum'], train['FloodProbability'], gridsize=50, cmap='YlOrRd', mincnt=1)
axes[1].set_xlabel('sum', fontsize=12, fontweight='bold')
axes[1].set_ylabel('FloodProbability', fontsize=12, fontweight='bold')
axes[1].set_title('Density Plot', fontsize=14, fontweight='bold')
plt.colorbar(axes[1].collections[0], ax=axes[1])

plt.tight_layout()
plt.savefig('eda_sum_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Magic Feature (от победителя!)

In [ ]:
# Magic feature: groupby('sum')['target'].std()
magic_std = train.groupby('sum')['FloodProbability'].std()

print(f'🪄 MAGIC FEATURE: groupby("sum")["FloodProbability"].std()')
print(f'   Уникальных sum: {len(magic_std)}')
print(f'   Диапазон std: [{magic_std.min():.6f}, {magic_std.max():.6f}]')
print(f'\nПримеры:')
print(magic_std.head(10))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(magic_std.index, magic_std.values, alpha=0.5, s=20)
axes[0].set_xlabel('sum', fontsize=12, fontweight='bold')
axes[0].set_ylabel('std(FloodProbability)', fontsize=12, fontweight='bold')
axes[0].set_title('Magic Feature по группам sum', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

axes[1].hist(magic_std.values, bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1].set_xlabel('std(FloodProbability)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Частота', fontsize=12, fontweight='bold')
axes[1].set_title('Распределение Magic Feature', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eda_magic_feature.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Корреляционный анализ

In [ ]:
# Корреляция признаков с таргетом
correlations = train[original_features + ['FloodProbability']].corr()['FloodProbability'].sort_values(ascending=False)

print('📊 КОРРЕЛЯЦИЯ С ТАРГЕТОМ:')
print(correlations)

# Визуализация
fig, ax = plt.subplots(figsize=(12, 8))
correlations[:-1].sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Корреляция', fontsize=12, fontweight='bold')
ax.set_title('Корреляция признаков с FloodProbability', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('eda_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Sorted Features (от победителя)

In [ ]:
# Sorted features
sorted_vals = np.sort(train[original_features].values, axis=1)
sorted_df = pd.DataFrame(sorted_vals, columns=[f'sorted_{i}' for i in range(len(original_features))])
sorted_df['FloodProbability'] = train['FloodProbability'].values

# Корреляция
sorted_corr = sorted_df.corr()['FloodProbability'].sort_values(ascending=False)

print('📊 КОРРЕЛЯЦИЯ SORTED FEATURES:')
print(sorted_corr[:-1])

# Визуализация
fig, ax = plt.subplots(figsize=(12, 6))
sorted_corr[:-1].plot(kind='bar', ax=ax, color='orange')
ax.set_xlabel('Sorted Feature', fontsize=12, fontweight='bold')
ax.set_ylabel('Корреляция', fontsize=12, fontweight='bold')
ax.set_title('Sorted Features vs FloodProbability', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('eda_sorted_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Count Features (от победителя)

In [ ]:
# Count features
count_features = pd.DataFrame()
for threshold in [6, 7, 8]:
    count_features[f'nb_sup{threshold}'] = (train[original_features] > threshold).sum(axis=1)
for threshold in [2, 3, 4]:
    count_features[f'nb_inf{threshold}'] = (train[original_features] < threshold).sum(axis=1)

count_features['FloodProbability'] = train['FloodProbability'].values

# Корреляция
count_corr = count_features.corr()['FloodProbability'].sort_values(ascending=False)

print('📊 КОРРЕЛЯЦИЯ COUNT FEATURES:')
print(count_corr[:-1])

# Визуализация
fig, ax = plt.subplots(figsize=(10, 6))
count_corr[:-1].plot(kind='bar', ax=ax, color='green')
ax.set_xlabel('Count Feature', fontsize=12, fontweight='bold')
ax.set_ylabel('Корреляция', fontsize=12, fontweight='bold')
ax.set_title('Count Features vs FloodProbability', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('eda_count_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Распределения признаков (Poisson?)

In [ ]:
# Распределения
fig, axes = plt.subplots(4, 5, figsize=(20, 12))
axes = axes.flatten()

for i, feat in enumerate(original_features):
    axes[i].hist(train[feat], bins=20, edgecolor='black', alpha=0.7)
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Распределения похожи на Poisson (как отметил победитель)')

## 8. Summary

In [ ]:
print('='*70)
print('🎯 EDA SUMMARY')
print('='*70)

print('\n📋 КЛЮЧЕВЫЕ НАХОДКИ (от победителя):')
print(f'   1. Target дискретизация: (target * 400).astype(int16)')
print(f'   2. sum — главный признак (R = {corr:.6f})')
print(f'   3. Magic feature: groupby(\'sum\')[\'target\'].std()')
print(f'   4. Sorted features полезны')
print(f'   5. Count features (nb_sup/nb_inf) полезны')
print(f'   6. Данные похожи на суммы Poisson')

print('\n📦 ФАЙЛЫ:')
print('   ✅ eda_target_distribution.png')
print('   ✅ eda_sum_analysis.png')
print('   ✅ eda_magic_feature.png')
print('   ✅ eda_correlations.png')
print('   ✅ eda_sorted_features.png')
print('   ✅ eda_count_features.png')
print('   ✅ eda_feature_distributions.png')

print('\n' + '='*70)
print('🚀 EDA COMPLETED!')
print('='*70)